# ResIPy Colab Notebook

This notebook sets up the environment to run ResIPy on Google Colab.
It clones the repository, installs necessary system dependencies (Wine, Gmsh) and Python packages.

## 1. Setup Environment
**Note:** Installing Wine takes a few minutes. This is required to run the inversion executables.

In [ ]:
import os
import shutil
import subprocess
from IPython import get_ipython

repo_dir = 'resipy'
repo_url = 'https://gitlab.com/hkex/resipy.git'
# Keep the package code, the package requirements, and the two example files used later in this notebook.
sparse_paths = [
    'requirements.txt',
    'src/resipy',
    'src/examples/dc-2d/syscal.csv',
    'src/examples/ip-2d/syscal.csv',
]

def run_command(cmd, cwd=None, quiet=False):
    stdout = subprocess.DEVNULL if quiet else None
    stderr = subprocess.DEVNULL if quiet else None
    subprocess.run(cmd, cwd=cwd, check=True, stdout=stdout, stderr=stderr)

required_paths = [os.path.join(repo_dir, path) for path in sparse_paths]

# 1. Clone only the files needed by this notebook
if not os.path.exists(repo_dir):
    print('Cloning a minimal ResIPy checkout...')
    try:
        run_command([
            'git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', repo_url, repo_dir
        ])
    except subprocess.CalledProcessError:
        shutil.rmtree(repo_dir, ignore_errors=True)
        print('Partial clone is unavailable here, falling back to a sparse shallow clone...')
        run_command(['git', 'clone', '--depth', '1', '--sparse', repo_url, repo_dir])

    run_command(['git', 'sparse-checkout', 'init', '--no-cone'], cwd=repo_dir, quiet=True)
    run_command(['git', 'sparse-checkout', 'set', *sparse_paths], cwd=repo_dir)
    print('Minimal checkout ready.')
else:
    print('ResIPy folder already exists.')
    if os.path.isdir(os.path.join(repo_dir, '.git')) and not all(os.path.exists(path) for path in required_paths):
        print('Adding required runtime files to the existing checkout...')
        run_command(['git', 'sparse-checkout', 'init', '--no-cone'], cwd=repo_dir, quiet=True)
        run_command(['git', 'sparse-checkout', 'set', *sparse_paths], cwd=repo_dir)

# 2. Ensure Colab uses NumPy < 2.0.0 before continuing
try:
    import numpy
    print(f"Current NumPy version: {numpy.__version__}")
    needs_numpy_restart = int(numpy.__version__.split('.')[0]) >= 2
except ImportError:
    print('NumPy is not installed in this runtime.')
    needs_numpy_restart = True

if needs_numpy_restart:
    print('Installing NumPy < 2.0.0 and restarting the runtime...')
    get_ipython().system('pip install "numpy<2.0.0" --quiet')
    os.kill(os.getpid(), 9)

print('NumPy is already compatible.')
print('Run the next cell after reconnecting to install Wine, Gmsh, and the remaining ResIPy Python dependencies.')

# 2. Install Wine, Gmsh, and remaining ResIPy Python dependencies
# Run this after the runtime reconnects if the previous cell restarted the kernel.

In [ ]:

import os
import shutil
import subprocess
import sys
import numpy

repo_dir = 'resipy'
requirements_file = os.path.join(repo_dir, 'requirements.txt')

def run_command(cmd, cwd=None, quiet=False):
    stdout = subprocess.DEVNULL if quiet else None
    stderr = subprocess.DEVNULL if quiet else None
    subprocess.run(cmd, cwd=cwd, check=True, stdout=stdout, stderr=stderr)

def find_wine_binary():
    candidate_names = ['wine', 'wine-stable', 'wine64', 'wine64-stable']
    for name in candidate_names:
        path = shutil.which(name)
        if path is not None:
            return path

    candidate_paths = [
        '/usr/bin/wine-stable',
        '/usr/bin/wine64-stable',
        '/usr/lib/wine/wine64',
        '/usr/lib/wine/wine',
    ]
    for path in candidate_paths:
        if os.path.exists(path) and os.access(path, os.X_OK):
            return path

    return None

def ensure_wine_launcher():
    wine_path = shutil.which('wine')
    if wine_path is not None:
        return wine_path

    backend_wine = find_wine_binary()
    if backend_wine is None:
        return None

    launcher_path = '/usr/local/bin/wine'
    with open(launcher_path, 'w', encoding='utf-8') as handle:
        handle.write(f'#!/bin/sh\nexec "{backend_wine}" "$@"\n')
    os.chmod(launcher_path, 0o755)
    return launcher_path

if int(numpy.__version__.split('.')[0]) >= 2:
    raise RuntimeError(
        'NumPy is still 2.x. Run the previous cell first and wait for the runtime to reconnect before continuing.'
    )

if not os.path.exists(requirements_file):
    raise FileNotFoundError(
        f'Missing {requirements_file}. Run the previous cell first to create the sparse checkout.'
    )

if find_wine_binary() is None:
    print('Installing Wine and Gmsh (this may take 2-3 minutes)...')
    run_command(['dpkg', '--add-architecture', 'i386'])
    run_command(['apt-get', 'update', '-qq'])
    run_command(
        ['apt-get', 'install', '-y', '--no-install-recommends', 'wine', 'wine32', 'wine64', 'gmsh'],
        quiet=True,
    )
    print('System dependencies installed.')
else:
    print('Wine is already installed.')

wine_path = ensure_wine_launcher()
if wine_path is None:
    raise RuntimeError(
        "Wine installation completed, but no runnable Wine executable was found. Hosted Colab may not support this backend for ResIPy's inversion step."
    )

wine_dir = os.path.dirname(wine_path)
path_entries = os.environ.get('PATH', '').split(os.pathsep)
if wine_dir not in path_entries:
    os.environ['PATH'] = wine_dir + os.pathsep + os.environ.get('PATH', '')

print(f'Using Wine launcher: {wine_path}')
run_command([wine_path, '--version'])

resipy_requirements = []
with open(requirements_file, encoding='utf-8') as handle:
    for raw_line in handle:
        requirement = raw_line.split('#', 1)[0].strip()
        if not requirement or requirement.lower().startswith('numpy'):
            continue
        resipy_requirements.append(requirement)

print('Installing remaining ResIPy Python dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *resipy_requirements], check=True)
print('Python dependencies installed.')
print('Continue with the next cell to verify the ResIPy import.')

# 3. Configure Path and Verify Import

In [ ]:
import os
import subprocess
import sys
import numpy

repo_dir = 'resipy'
resipy_src = os.path.abspath(os.path.join(repo_dir, 'src'))
requirements_file = os.path.join(repo_dir, 'requirements.txt')

if not os.path.exists(resipy_src):
    raise FileNotFoundError(
        f'Missing {resipy_src}. Run Cell 2 first to create the sparse checkout.'
    )

if resipy_src not in sys.path:
    sys.path.insert(0, resipy_src)

try:
    import resipy
except ModuleNotFoundError as exc:
    if exc.name in {None, 'resipy'}:
        raise

    if not os.path.exists(requirements_file):
        raise FileNotFoundError(
            f'Missing {requirements_file}. Run Cell 2 and Cell 3 first, then retry this cell.'
        ) from exc

    resipy_requirements = []
    with open(requirements_file, encoding='utf-8') as handle:
        for raw_line in handle:
            requirement = raw_line.split('#', 1)[0].strip()
            if not requirement or requirement.lower().startswith('numpy'):
                continue
            resipy_requirements.append(requirement)

    print(f"Missing Python package detected during ResIPy import: {exc.name}")
    print('Installing cloned ResIPy requirements and retrying import...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *resipy_requirements], check=True)

    stale_modules = [name for name in sys.modules if name == 'resipy' or name.startswith('resipy.')]
    for name in stale_modules:
        sys.modules.pop(name, None)

    import resipy

print(f"ResIPy imported from: {resipy.__file__}")
print(f"NumPy version: {numpy.__version__}")
print(f"NumPy loaded from: {numpy.__file__}")
print(f"Python version: {sys.version}")

## 3.1 Basic Inversion Formulation

For the DC inversions used in this notebook, ResIPy calls `R2` and solves a regularized inverse problem that balances fitting the observed data and keeping the recovered model smooth and stable.

The basic objective function can be written as:

$$
\Phi(\mathbf{m}) = \Phi_d(\mathbf{m}) + \lambda \, \Phi_m(\mathbf{m})
$$

where the data-misfit term is

$$
\Phi_d(\mathbf{m}) = \left\| \mathbf{W}_d \left( \mathbf{d}^{obs} - \mathbf{g}(\mathbf{m}) \right) \right\|_2^2
$$

and the model regularization term is

$$
\Phi_m(\mathbf{m}) = \left\| \mathbf{W}_m \left( \mathbf{m} - \mathbf{m}_{ref} \right) \right\|_2^2
$$

Here:

- $\mathbf{d}^{obs}$ is the observed resistivity data
- $\mathbf{g}(\mathbf{m})$ is the forward-model response predicted from the current model
- $\mathbf{m}$ is the subsurface model being solved for
- $\mathbf{W}_d$ is the data-weighting matrix built from the estimated data errors
- $\mathbf{W}_m$ is the smoothing or roughness operator
- $\mathbf{m}_{ref}$ is the starting or reference model
- $\lambda$ controls the trade-off between fitting the data and keeping the model smooth

In linearized form, one inversion update is commonly written as:

$$
\left( \mathbf{J}_k^T \mathbf{W}_d^T \mathbf{W}_d \mathbf{J}_k + \lambda \, \mathbf{W}_m^T \mathbf{W}_m \right) \Delta \mathbf{m}_k
=
\mathbf{J}_k^T \mathbf{W}_d^T \mathbf{W}_d \left[ \mathbf{d}^{obs} - \mathbf{g}(\mathbf{m}_k) \right]
$$

where $\mathbf{J}_k$ is the Jacobian or sensitivity matrix at iteration $k$, and the model is updated by

$$
\mathbf{m}_{k+1} = \mathbf{m}_k + \Delta \mathbf{m}_k
$$

In practical terms, the inversion shown later in this notebook is trying to reduce the weighted data misfit while avoiding an unrealistically rough resistivity model. The RMS values reported during inversion are a compact summary of how well the current model matches the observed data.

# 4. Run Example: 2D DC Inversion

This example demonstrates how to load data, view pseudo-section, and perform inversion.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from resipy import Project

# Path to example data
testdir = 'resipy/src/examples/dc-2d/'

# Create Project object
k = Project(typ='R2')

# Create Survey from CSV file
print("Loading survey data...")
k.createSurvey(testdir + 'syscal.csv', ftype='Syscal')

# Show Pseudo Section
print("Plotting Pseudo Section...")
k.showPseudo()

# Run Inversion
# This will use R2.exe via Wine
print("Running Inversion (this may take a moment)...")
k.invert(iplot=False)

# Show Results
print("Displaying Results...")
k.showResults(attr='Resistivity(ohm.m)', sens=False, contour=True)

# 5. Geophysical mappoing of hyporheic process

This is a field data set dedicated to study the Hyporheic exchange and its influences on water quality and controls over numerous physical, chemical,
and biological processes (McGarr1, et al., 2021 DOI: 10.1002/hyp.14358).

By using ERT and recording the data for a period of time, the hyporheic process is able to be visualized in terms of the changes of resistivity distribution.

<img src="https://github.com/uqcivil/ERTH3250-Geophysics-Inversion/blob/main/hyporheic1.png?raw=1" width=65%>

(a) Example compound bar depicting sedimentological features at multiple scales; (b) & (c) Compound bar at the
Theis Environmental Monitoring and Modeling Site (2019)/(1945) located in southwestern Ohio within the Great Miami River watershed. The red shaded
area with a black outline is the modern floodplain on the northern side of the river. The area outlined in red is the cross-bar channel examined in the
study.



<img src="https://github.com/uqcivil/ERTH3250-Geophysics-Inversion/blob/main/hyporheic2.png?raw=1" width=65%>

EMI range and ERT survey grid shown in larger image. Borehole
locations and ERT grid shown in the zoomed image.

<img src="https://github.com/uqcivil/ERTH3250-Geophysics-Inversion/blob/main/hyporheic3.png?raw=1" width=45%>

Inverted ERT resulting reisistivity through different days.

## Goal is to replicate the resistivity inversion models and develop further with a moisture content distribution model.

In [ ]:
import os
import shutil
import subprocess

data_repo_dir = 'ERTH3250-Geophysics-Inversion'
data_repo_url = 'https://github.com/uqcivil/ERTH3250-Geophysics-Inversion.git'
data_sparse_path = 'TEMMS CB ERT'
data_dir = os.path.join(data_repo_dir, data_sparse_path)

def run_command(cmd, cwd=None, quiet=False):
    stdout = subprocess.DEVNULL if quiet else None
    stderr = subprocess.DEVNULL if quiet else None
    subprocess.run(cmd, cwd=cwd, check=True, stdout=stdout, stderr=stderr)

if not os.path.exists(data_repo_dir):
    print('Cloning the TEMMS CB ERT data folder...')
    try:
        run_command([
            'git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', data_repo_url, data_repo_dir
        ])
    except subprocess.CalledProcessError:
        shutil.rmtree(data_repo_dir, ignore_errors=True)
        print('Partial clone is unavailable here, falling back to a sparse shallow clone...')
        run_command(['git', 'clone', '--depth', '1', '--sparse', data_repo_url, data_repo_dir])
else:
    print('Data repository folder already exists.')

if not os.path.isdir(os.path.join(data_repo_dir, '.git')):
    raise FileNotFoundError(
        f'Missing git metadata in {data_repo_dir}. Delete that folder and rerun this cell.'
    )

run_command(['git', 'sparse-checkout', 'init', '--no-cone'], cwd=data_repo_dir, quiet=True)
run_command(['git', 'sparse-checkout', 'set', data_sparse_path], cwd=data_repo_dir)

if not os.path.exists(data_dir):
    raise FileNotFoundError(f'Expected data folder not found: {data_dir}')

print(f'Data downloaded to: {os.path.abspath(data_dir)}')
print('Available files:')
for name in sorted(os.listdir(data_dir)):
    print(f' - {name}')

In [ ]:
import contextlib
import io
import os
import uuid

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
from IPython.display import display, update_display
from resipy import Project

target_name = 'mcline3-070720.Dat'
candidate_paths = [
    os.path.join('ERTH3250-Geophysics-Inversion', 'TEMMS CB ERT', '070720', target_name),
    os.path.join('TEMMS CB ERT', '070720', target_name),
]

data_file = None
for candidate in candidate_paths:
    if os.path.exists(candidate):
        data_file = candidate
        break

if data_file is None:
    for root, _, files in os.walk('.'):
        if target_name in files:
            data_file = os.path.join(root, target_name)
            break

if data_file is None:
    raise FileNotFoundError(
        'Could not find mcline2-070720.Dat. Run the TEMMS data download cell first.'
    )

# Editable mesh settings for this survey.
characteristic_length = -1  # ResIPy `cl`; keep -1 for automatic sizing.
refine_factor = 2.0  # ResIPy `cl_factor`; controls mesh expansion away from electrodes.
fine_mesh_depth = None  # ResIPy `fmd`; keep None to use the automatic fine-mesh depth.
mesh_refine_iterations = 0  # ResIPy `refine`; extra refinement passes after mesh generation.
outer_boundary_factor = None  # Optional ResIPy/triMesh `cln_factor`.
crop_below_fmd = False  # If True, clip the live model view below the fine-mesh depth.

# Editable inversion settings for this survey.
a_wgt = 0.001
b_wgt = 0.02
max_iterations = 8
tolerance = 1.0
target_decrease = None  # Optional ResIPy `target_decrease`.
min_error = None  # Optional ResIPy `min_error`.

mesh_settings = {
    'typ': 'trian',
    'cl': characteristic_length,
    'cl_factor': refine_factor,
    'fmd': fine_mesh_depth,
    'refine': mesh_refine_iterations,
}
if outer_boundary_factor is not None:
    mesh_settings['cln_factor'] = outer_boundary_factor

inversion_settings = {
    'a_wgt': a_wgt,
    'b_wgt': b_wgt,
    'max_iterations': max_iterations,
    'tolerance': tolerance,
}
if target_decrease is not None:
    inversion_settings['target_decrease'] = target_decrease
if min_error is not None:
    inversion_settings['min_error'] = min_error

print(f'Using data file: {os.path.abspath(data_file)}')

k_temms = Project(typ='R2')
print('Loading TEMMS survey data...')
k_temms.createSurvey(data_file, ftype='ResInv')

print('Plotting pseudo section...')
k_temms.showPseudo()

print('Creating mesh with custom settings...')
print(mesh_settings)
k_temms.createMesh(**mesh_settings)

rms_values = []
live_state = {
    'last_line': 'Waiting for inversion output...',
    'display_initialized': False,
}
display_id = f"temms-live-{uuid.uuid4().hex}"
live_colorbar_label = r'$\log_{10}\rho$ [$\Omega$.m]'

def plot_latest_iteration(ax):
    iteration_files = sorted(
        file_name
        for file_name in os.listdir(k_temms.dirname)
        if file_name.endswith('_res.dat') and len(file_name) in {12, 16}
    )

    if len(iteration_files) <= 1:
        return None

    latest_iteration_file = os.path.join(k_temms.dirname, iteration_files[-2])
    iteration_data = pd.read_csv(latest_iteration_file, header=None, sep=r'\s+').values

    if iteration_data.shape[0] == 0:
        return None

    triangulation = mtri.Triangulation(iteration_data[:, 0], iteration_data[:, 1])
    if k_temms.typ[0] == 'c':
        plotted_values = iteration_data[:, 4]
    else:
        plotted_values = iteration_data[:, 3]

    contour_set = ax.tricontourf(triangulation, plotted_values, extend='both', cmap='viridis')

    if (k_temms.topo.shape[0] != 0) or (k_temms.elec['buried'].sum() < k_temms.elec.shape[0]):
        k_temms._clipContour(ax, contour_set, cropMaxDepth=crop_below_fmd)

    elec = k_temms.elec[~k_temms.elec['remote']][['x', 'z']].values
    ax.plot(elec[:, 0], elec[:, 1], 'ko', markersize=4)
    ax.set_aspect('equal')
    ax.set_xlabel('Distance [m]')
    ax.set_ylabel('Elevation [m]')
    ax.set_xlim([np.min(elec[:, 0]), np.max(elec[:, 0])])
    ax.set_ylim(k_temms.zlim)
    ax.set_title('Current iteration model')

    return contour_set

def render_live_view(final=False):
    fig, (ax_rms, ax_model) = plt.subplots(1, 2, figsize=(14, 5))
    fig.subplots_adjust(wspace=0.35, top=0.9, bottom=0.12)

    ax_rms.set_title('RMS misfit by iteration')
    ax_rms.set_xlabel('Iteration')
    ax_rms.set_ylabel('RMS')
    ax_rms.grid(True, alpha=0.3)

    if rms_values:
        x_values = list(range(1, len(rms_values) + 1))
        ax_rms.plot(x_values, rms_values, 'o-', color='tab:blue')
        ax_rms.scatter(x_values[-1], rms_values[-1], color='tab:red', zorder=3)
    else:
        ax_rms.text(
            0.5,
            0.5,
            'Waiting for RMS updates...',
            transform=ax_rms.transAxes,
            ha='center',
            va='center',
        )

    if final and len(k_temms.meshResults) > 0:
        with contextlib.redirect_stdout(io.StringIO()):
            k_temms.showResults(
                ax=ax_model,
                attr='Resistivity(log10)',
                sens=False,
                contour=True,
                cropMaxDepth=crop_below_fmd,
            )
    else:
        contour_set = plot_latest_iteration(ax_model)
        if contour_set is not None:
            fig.colorbar(
                contour_set,
                ax=ax_model,
                label=live_colorbar_label,
            )
        else:
            ax_model.set_title('Current iteration model')
            ax_model.set_xlabel('Distance [m]')
            ax_model.set_ylabel('Elevation [m]')
            ax_model.text(
                0.5,
                0.5,
                'Waiting for first completed iteration...',
                transform=ax_model.transAxes,
                ha='center',
                va='center',
            )

    fig.suptitle(live_state['last_line'], fontsize=12, y=0.97)

    if live_state['display_initialized']:
        update_display(fig, display_id=display_id)
    else:
        display(fig, display_id=display_id)
        live_state['display_initialized'] = True

    plt.close(fig)

def live_dump(text):
    stripped = text.strip()
    if stripped:
        live_state['last_line'] = stripped

    tokens = stripped.split()
    if tokens:
        if tokens[0] == 'Initial':
            try:
                rms_values.append(float(tokens[3]))
            except (IndexError, ValueError):
                pass
            render_live_view()
        elif tokens[0] == 'Iteration':
            render_live_view()
        elif tokens[0] in {'Final', 'End', 'FATAL:'}:
            render_live_view()

        if tokens[0] in {'Initial', 'Iteration', 'Processing', 'Final', 'End', 'FATAL:'}:
            print(stripped)

render_live_view()

print('Running inversion with live RMS/model updates (this may take a moment)...')
print(inversion_settings)
k_temms.invert(param=inversion_settings, iplot=False, dump=live_dump)

live_state['last_line'] = (
    f"Final RMS misfit: {k_temms.pinfo.get('RMS of inverse solution', 'N/A')} after "
    f"{k_temms.pinfo.get('Number of iterations', 'N/A')} iterations"
)
render_live_view(final=True)

print(live_state['last_line'])
print('Plotting final mesh...')
k_temms.showMesh()

print('Displaying final inversion results...')
fig_final, ax_final = plt.subplots(figsize=(8, 5.5))
fig_final.subplots_adjust(top=0.9, bottom=0.12)
with contextlib.redirect_stdout(io.StringIO()):
    k_temms.showResults(
        ax=ax_final,
        attr='Resistivity(ohm.m)',
        sens=False,
        contour=True,
    )
plt.show()

# 6. Meaning of `a_wgt` and `b_wgt` in the DC inversion

> This notebook uses `Project(typ='R2')`, so the equations below are for the DC weighting used by `R2`.

For each datum $i$, ResIPy uses `a_wgt` and `b_wgt` to define the data-error variance as:

$$
\sigma_i^2 = a_{\mathrm{wgt}}^2 + b_{\mathrm{wgt}}^2 R_i^2
$$

so the corresponding standard deviation is:

$$
\sigma_i = \sqrt{a_{\mathrm{wgt}}^2 + \left(b_{\mathrm{wgt}} R_i\right)^2}
$$

where:

- $R_i$ is the observed resistance or transfer resistance for datum $i$
- $a_{\mathrm{wgt}}$ is the absolute error floor
- $b_{\mathrm{wgt}}$ is the relative error coefficient

These errors then enter the weighted data-misfit term used by the inversion as:

$$
\Phi_d = \sum_{i=1}^{N} \left(\frac{R_i^{\mathrm{obs}} - R_i^{\mathrm{calc}}}{\sigma_i}\right)^2
$$

After substituting the error model, this becomes:

$$
\Phi_d = \sum_{i=1}^{N} \frac{\left(R_i^{\mathrm{obs}} - R_i^{\mathrm{calc}}\right)^2}{a_{\mathrm{wgt}}^2 + b_{\mathrm{wgt}}^2 \left(R_i^{\mathrm{obs}}\right)^2}
$$

So increasing either `a_wgt` or `b_wgt` increases the denominator, reduces the weight of the data in the misfit term, and usually makes the inversion less aggressive in fitting the observations.

The full inversion objective is the usual combination of data misfit and model regularization:

$$
\Phi = \Phi_d + \lambda \Phi_m
$$

where $\Phi_m$ is the model roughness or regularization term and $\lambda$ is the regularization strength selected internally by the inversion workflow.

Note:

- If you provide individual per-datum errors through `resError`, those can override the default `a_wgt` and `b_wgt` weighting.
- In ResIPy, the local Python-side error estimate for this DC case follows the same variance form shown above before the values are passed into `R2`.

# 7. Moisture Content derived from resistivity and chargeability using Dynamic Stern Layer (DSL) Model

## Moisture Content Derived from Resistivity and Chargeability Using the Dynamic Stern Layer (DSL) Model

The Dynamic Stern Layer (DSL) model developed by :contentReference[oaicite:0]{index=0} relates electrical conductivity and induced polarization to the hydrogeological properties of porous media. The model links polarization effects to the cation exchange capacity (CEC), pore water conductivity, and volumetric water content. The DSL framework is widely used in spectral induced polarization (SIP) and time-domain induced polarization (TDIP) studies for estimating moisture content and permeability from geophysical inversion results.

The volumetric water content is defined as

$$
\theta = \frac{V_w}{V_b}
$$

where:

- $\theta$ = volumetric water content (dimensionless or m$^3$/m$^3$)
- $V_w$ = volume of water
- $V_b$ = bulk volume

In hydrology and geophysics, $\theta$ is commonly referred to as the volumetric moisture content.

According to the DSL model, the low-frequency conductivity is expressed as

$$
\sigma_0 = \theta^{m}\sigma_w + \theta^{m-1}\rho_g B \, \mathrm{CEC}
$$

and the high-frequency conductivity is

$$
\sigma_{\infty} = \theta^{m}\sigma_w + \theta^{m-1}\rho_g (B + \lambda)\,\mathrm{CEC}
$$

where:

- $\sigma_0$ = DC conductivity (S/m)
- $\sigma_{\infty}$ = high-frequency conductivity (S/m)
- $\sigma_w$ = pore water conductivity (S/m)
- $m$ = cementation exponent
- $\rho_g$ = grain density (kg/m$^3$)
- $B$ = apparent mobility for surface conduction (m$^2$ s$^{-1}$ V$^{-1}$)
- $\lambda$ = apparent mobility associated with polarization (m$^2$ s$^{-1}$ V$^{-1}$)
- CEC = cation exchange capacity (C/kg)

The normalized chargeability is defined as

$$
M_n = \sigma_{\infty} - \sigma_0
$$

Substituting the conductivity equations gives

$$
M_n = \theta^{m-1}\rho_g \lambda \, \mathrm{CEC}
$$

This relationship shows that normalized chargeability is directly proportional to the volumetric moisture content, grain density, polarization mobility, and cation exchange capacity. :contentReference[oaicite:2]{index=2}

Combining the conductivity and normalized chargeability equations allows the volumetric moisture content to be estimated directly from inversion results:

$$
\theta =
\left[
\frac{1}{\sigma_w}
\left(
\sigma_{\infty} - \frac{M_n}{R}
\right)
\right]^{1/m}
$$

where

$$
R = \frac{\lambda}{B}
$$

is the ratio between polarization mobility and surface conductivity mobility.

This formulation enables SIP and TDIP inversion results to be transformed into physically meaningful hydrogeological properties such as moisture content, CEC, and permeability. :contentReference[oaicite:3]{index=3}

## References

1. Revil, A. et al. (2021). *Field-scale estimation of soil properties from spectral induced polarization tomography*. Geochimica et Cosmochimica Acta.

2. Revil, A. et al. (2020). *Induced polarization as a tool to non-intrusively characterize embankment hydraulic properties*. Engineering Geology.

3. Revil, A. et al. (2017). *Spectral induced polarization of volcanic rocks*. Geophysical Journal International.

4. Revil, A. and Jardani, A. (2013). *The Self-Potential Method: Theory and Applications in Environmental Geosciences*. Cambridge University Press.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from resipy import Project


def load_resistivity_result(project=None, inversion_dir=None, project_file=None, survey_index=0, typ='R2'):
    """Load inversion resistivity values from an in-memory Project, inversion folder, or .resipy file."""
    if project is None:
        project = Project(typ=typ)
        if inversion_dir:
            project.loadResults(inversion_dir)
        elif project_file:
            project.loadProject(project_file)
        else:
            raise ValueError(
                'Provide one of: project (already inverted), inversion_dir (R2 output folder), or project_file (.resipy).'
            )

    if len(project.meshResults) == 0:
        raise RuntimeError('No inversion results found in project.meshResults.')

    if survey_index < 0 or survey_index >= len(project.meshResults):
        raise IndexError(f'survey_index={survey_index} is outside valid range [0, {len(project.meshResults)-1}].')

    mesh = project.meshResults[survey_index]
    resistivity_candidates = ['Resistivity(ohm.m)', 'Resistivity(Ohm-m)', 'Resistivity']

    for column in resistivity_candidates:
        if column in mesh.df.columns:
            return project, mesh, mesh.df[column].to_numpy(dtype=float), column

    raise KeyError(
        f'No resistivity column found. Available columns: {list(mesh.df.columns)}'
    )


def compute_theta_dsl(resistivity_ohm_m, sigma_w=1.0, R=0.09, m=2.0, M=0.01):
    """Compute volumetric moisture content theta from resistivity and chargeability using the DSL relation."""
    if sigma_w <= 0:
        raise ValueError('sigma_w must be > 0.')
    if R <= 0:
        raise ValueError('R must be > 0.')
    if m <= 0:
        raise ValueError('m must be > 0.')
    if not (0 < M < 1):
        raise ValueError('M must be between 0 and 1.')

    resistivity_ohm_m = np.asarray(resistivity_ohm_m, dtype=float)
    sigma_0 = 1.0 / resistivity_ohm_m
    Mn = (M / (1.0 - M)) * sigma_0
    sigma_infinite = sigma_0 + Mn

    theta_term = (sigma_infinite - (Mn / R)) / sigma_w
    theta = np.where(theta_term > 0, theta_term ** (1.0 / m), np.nan)
    return theta


# Option A (default): use the project created by the inversion cell above (k_temms).
# Option B: set inversion_dir to a folder containing protocol.dat, electrodes.dat, mesh.msh, f001_res.vtk...
# Option C: set project_file to a saved .resipy project file.
inversion_dir = None
project_file = None
survey_index = 0

project_ref = globals().get('k_temms', None)
if project_ref is None and inversion_dir is None and project_file is None:
    raise ValueError(
        'k_temms was not found. Run the inversion cell first, or set inversion_dir/project_file in this cell.'
    )

project_ref, mesh_result, resistivity_values, resistivity_column = load_resistivity_result(
    project=project_ref,
    inversion_dir=inversion_dir,
    project_file=project_file,
    survey_index=survey_index,
)

# DSL parameters (adjust as needed for site-specific calibration)
rho_g = 2.65e3  # [kg/m^3] kept for completeness from the DSL parameter set
sigma_w = 1.0   # [S/m]
R = 0.09
m = 2.0
M = 0.01

theta = compute_theta_dsl(
    resistivity_values,
    sigma_w=sigma_w,
    R=R,
    m=m,
    M=M,
)

theta_attr = 'MoistureContent(theta)'
if theta_attr in mesh_result.df.columns:
    mesh_result.df[theta_attr] = theta
else:
    mesh_result.addAttribute(theta, theta_attr)

valid = np.isfinite(theta)
print(f'Resistivity source column: {resistivity_column}')
print(f'Theta valid cells: {valid.sum()}/{theta.size}')
if valid.any():
    print(f'Theta range: min={np.nanmin(theta):.4f}, max={np.nanmax(theta):.4f}')

fig = mesh_result.show(
    attr=theta_attr,
    color_map='YlGnBu',
    contour=True,
    sens=False,
    electrodes=True,
    clabel='Volumetric moisture content theta [-]',
)
plt.title('Moisture content (theta) from DSL relation')
plt.show()